# Prática — Módulo 8 – LDA

In [22]:
#  Pequeno corpus de exemplo
docs = [
    "o time venceu o campeonato com muitos gols e boa defesa",
    "o jogador marcou dois gols na partida final",
    "o técnico mudou a estratégia do time no segundo tempo",
    "a eleição presidencial gerou debate entre candidatos e partidos",
    "o congresso aprovou uma nova proposta de governo",
    "os políticos discutiram economia e reforma tributária"
]

In [23]:
# Transformar o corpus em matriz de contagem Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

In [24]:
# Visualizar a matriz Bag of Words
import pandas as pd

df_bow = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df_bow

,aprovou,boa,campeonato,candidatos,com,congresso,de,debate,defesa,discutiram,...,presidencial,proposta,reforma,segundo,tempo,time,tributária,técnico,uma,venceu
0,0,1,1,0,1,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,1,0,1,0,0
3,0,0,0,1,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,1,1,0,0,0,...,0,1,0,0,0,0,0,0,1,0
5,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,1,0,0,0


## Pergunta
1. Porque existem 6 linhas?
2. Porque cada linha possui 41 colunas?

In [25]:
# Treinar o LDA
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X)

LatentDirichletAllocation(n_components=3, random_state=42)

## Pergunta
1. O que o modelo LDA recebe como entrada?
2. O modelo já conhece os tópicos antes do treinamento?
3. O que o LDA vai tentar aprender a partir dessa matriz?
4. O que é o parâmetro n_components?


In [26]:
# Visualizar as principais palavras de cada tópico
def mostrar_topicos(modelo, palavras, n_top=5):
    for i, topic in enumerate(modelo.components_):
        top_idx = topic.argsort()[::-1][:n_top]
        top_words = [palavras[j] for j in top_idx]
        print(f"Tópico {i+1}: {', '.join(top_words)}")

mostrar_topicos(lda, vectorizer.get_feature_names_out())

Tópico 1: time, segundo, técnico, tempo, no
Tópico 2: gols, defesa, boa, campeonato, com
Tópico 3: uma, proposta, aprovou, governo, de


In [27]:
# Objetivo: mostrar palavras + pesos
for i, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[::-1][:5]
    for j in top_idx:
        print(f"Tópico {i+1}: {vectorizer.get_feature_names_out()[j]} ({topic[j]:.2f})")

Tópico 1: time (1.33)
Tópico 1: segundo (1.33)
Tópico 1: técnico (1.33)
Tópico 1: tempo (1.33)
Tópico 1: no (1.33)
Tópico 2: gols (2.33)
Tópico 2: defesa (1.33)
Tópico 2: boa (1.33)
Tópico 2: campeonato (1.33)
Tópico 2: com (1.33)
Tópico 3: uma (1.33)
Tópico 3: proposta (1.33)
Tópico 3: aprovou (1.33)
Tópico 3: governo (1.33)
Tópico 3: de (1.33)


## Pergunta
1. O que acontece com os tópicos se eu mudar o valor random_state=42?
2. Os tópicos listados são tópicos dos documentos ou dos corpus inteiro?

In [28]:
# Ver mistura de tópicos em cada documento
doc_topic = lda.transform(X)

for i, dist in enumerate(doc_topic):
    print(f"Doc {i+1}: {dist}")

Doc 1: [0.037964   0.92475407 0.03728194]
Doc 2: [0.0419408 0.9161184 0.0419408]
Doc 3: [0.92479986 0.03791008 0.03729006]
Doc 4: [0.04195299 0.04194249 0.91610453]
Doc 5: [0.04195299 0.04194249 0.91610453]
Doc 6: [0.90411681 0.04793557 0.04794762]


## Pergunta
1. Qual o somatório dos valores em cada vetor do documento?

In [29]:
# Mostrar o tópico principal de cada documento
for i, dist in enumerate(doc_topic):
    print(f"Doc {i+1}: tópico dominante = {dist.argmax()+1}")

Doc 1: tópico dominante = 2
Doc 2: tópico dominante = 2
Doc 3: tópico dominante = 1
Doc 4: tópico dominante = 3
Doc 5: tópico dominante = 3
Doc 6: tópico dominante = 1


**# Parte 02**

In [30]:
# Carregar o CSV em um DataFrame
import pandas as pd

df = pd.read_csv("/content/Mental-Health-Twitter.csv")

In [31]:
# Objetivo: visualizar estrutura do dataset
df.head()

,Unnamed: 0,post_id,post_created,post_text,user_id,followers,friends,favourites,statuses,retweets,label
0,0,637894677824413696,Sun Aug 30 07:48:37 +0000 2015,It's just over 2 years since I was diagnosed w...,1013187241,84,211,251,837,0,1
1,1,637890384576778240,Sun Aug 30 07:31:33 +0000 2015,"It's Sunday, I need a break, so I'm planning t...",1013187241,84,211,251,837,1,1
2,2,637749345908051968,Sat Aug 29 22:11:07 +0000 2015,Awake but tired. I need to sleep but my brain ...,1013187241,84,211,251,837,0,1
3,3,637696421077123073,Sat Aug 29 18:40:49 +0000 2015,RT @SewHQ: #Retro bears make perfect gifts and...,1013187241,84,211,251,837,2,1
4,4,637696327485366272,Sat Aug 29 18:40:26 +0000 2015,It’s hard to say whether packing lists are mak...,1013187241,84,211,251,837,1,1


## Pergunta
1. Qual coluna contém o texto?

In [32]:
# Extrair os documentos (somente a coluna de texto)
docs = df["post_text"].dropna().tolist()

In [33]:
# Transformar textos reais em matriz Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(docs)

In [34]:
# Ver o volume dados (dimensões da matriz)
print(X.shape)

(20000, 32134)


## Pergunta
1. Quantos documentos existem?
2. Qual é o tamanho do vocabulário?

In [35]:
# Treinar o modelo LDA
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

LatentDirichletAllocation(n_components=5, random_state=42)

# Pergunta
1. O treinamento demorou mais agora?

In [36]:
# Ver as principais palavras de cada tópico
import numpy as np

for i, topic in enumerate(lda.components_):
    prob = topic / topic.sum()
    top_idx = prob.argsort()[::-1][:10]
    print(f"Tópico {i+1}: " + ", ".join([f"{vectorizer.get_feature_names_out()[j]} ({prob[j]:.3f})" for j in top_idx]))

Tópico 1: https (0.058), http (0.028), depression (0.021), rt (0.016), user (0.009), treatments (0.006), overcome (0.006), health (0.004), mental (0.003), morning (0.003)
Tópico 2: rt (0.035), http (0.023), https (0.011), love (0.009), like (0.007), just (0.006), know (0.006), yong (0.006), day (0.005), really (0.005)
Tópico 3: rt (0.021), just (0.012), https (0.011), like (0.010), misslusyd (0.010), http (0.009), don (0.009), genevieveverso (0.006), thefuxedos (0.006), azarkansero (0.006)
Tópico 4: http (0.030), https (0.027), bit (0.018), ly (0.017), say (0.010), twitter (0.009), thank (0.009), rt (0.008), following (0.007), hello (0.007)
Tópico 5: https (0.050), rt (0.016), trump (0.009), amp (0.006), people (0.005), putin (0.004), like (0.004), did (0.004), 000 (0.003), mnwild (0.003)


## Pergunta
1. Por que aparecem termos como “https” e “rt”?

In [37]:
# Limpar texto (remover urls, RT, caracteres estranhos) + stopwords
import re
from sklearn.feature_extraction.text import CountVectorizer

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"http\S+|www\S+", "", texto)  # remove URLs
    texto = re.sub(r"\brt\b", "", texto)          # remove RT
    texto = re.sub(r"@\w+", "", texto)            # remove @user
    texto = re.sub(r"[^a-z\s]", "", texto)        # remove números/pontuação
    return texto

docs_limpos = [limpar_texto(t) for t in docs]

vectorizer = CountVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.9
)

X = vectorizer.fit_transform(docs_limpos)

In [38]:
# Treinar - novamentre - com dados limpos
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

LatentDirichletAllocation(n_components=5, random_state=42)

In [39]:
# Ver as principais palavras de cada tópico (após a limpeza)
import numpy as np

for i, topic in enumerate(lda.components_):
    prob = topic / topic.sum()
    top_idx = prob.argsort()[::-1][:10]
    print(f"Tópico {i+1}: " + ", ".join([f"{vectorizer.get_feature_names_out()[j]} ({prob[j]:.3f})" for j in top_idx]))

Tópico 1: depression (0.045), yong (0.017), treatments (0.017), thanks (0.014), best (0.014), overcome (0.013), hey (0.012), follow (0.012), video (0.009), good (0.009)
Tópico 2: like (0.022), love (0.015), people (0.013), just (0.010), depression (0.009), make (0.009), help (0.008), mental (0.008), look (0.007), need (0.007)
Tópico 3: im (0.050), dont (0.023), like (0.019), just (0.014), people (0.012), fuck (0.010), ive (0.009), know (0.008), love (0.008), trump (0.008)
Tópico 4: just (0.021), know (0.013), day (0.012), oh (0.011), want (0.010), amp (0.009), got (0.009), happy (0.008), dont (0.008), youre (0.008)
Tópico 5: thank (0.026), say (0.023), twitter (0.021), following (0.015), hello (0.014), good (0.014), way (0.011), girl (0.009), morning (0.009), anytime (0.009)


In [40]:
# Ver mistura de tópicos em cada documento (apenas os 10 primeiros...)
doc_topic = lda.transform(X)

for i, dist in enumerate(doc_topic[:5]):
    print(f"Doc {i+1}: {docs[i][:200]}...")
    print(f"          {dist}")

Doc 1: It's just over 2 years since I was diagnosed with #anxiety and #depression. Today I'm taking a moment to reflect on how far I've come since....
          [0.01581274 0.57391453 0.14090844 0.25391778 0.01544652]
Doc 2: It's Sunday, I need a break, so I'm planning to spend as little time as possible on the #A14......
          [0.02009892 0.02035123 0.11180288 0.82753472 0.02021225]
Doc 3: Awake but tired. I need to sleep but my brain has other ideas......
          [0.88244071 0.02977629 0.03000371 0.02910122 0.02867807]
Doc 4: RT @SewHQ: #Retro bears make perfect gifts and are great for beginners too! Get stitching with October's Sew on sale NOW! #yay http://t.co/…...
          [0.02938188 0.43047018 0.0288014  0.3177872  0.19355934]
Doc 5: It’s hard to say whether packing lists are making life easier or just reinforcing how much still needs doing... #movinghouse #anxiety...
          [0.01846263 0.1609002  0.28994117 0.25327571 0.2774203 ]


## Pergunta
1. O tópico dominante de cada documento condiz com o conteúdo do texto?
2. O que você faria como analista para melhorar a qualidade dos tópicos?